<a href="https://colab.research.google.com/github/cactus1386/NationalCard-ImageProccessing/blob/main/main(use_flag_logo).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and import Libraries

In [1]:
! pip install ultralytics easyocr hezar insightface onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 18.1 MB/s eta 0:00:00
  Installing build dependencies ... canceledERROR: Operation cancelled by user


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import os
import pandas as pd
import re
from hezar.models import Model
from insightface.app import FaceAnalysis

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Read kibord dataset

In [ ]:
import gdown
import zipfile
import os

In [ ]:
file_id = "1EP7JhNFoLtlmJCabhrpTYOkMJ2c_OyMM"
dataset_zip_path = "/content/dataset.zip"
extract_path = "/content/dataset"

In [ ]:
gdown.download(f"https://drive.google.com/uc?id={file_id}", dataset_zip_path, quiet=False)

with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset downloaded and extracted successfully!")


Downloading...
From (original): https://drive.google.com/uc?id=1EP7JhNFoLtlmJCabhrpTYOkMJ2c_OyMM
From (redirected): https://drive.google.com/uc?id=1EP7JhNFoLtlmJCabhrpTYOkMJ2c_OyMM&confirm=t&uuid=6f05fa74-3d7e-462a-99b3-fd8490a30692
To: /content/dataset.zip
100%|██████████| 539M/539M [00:10<00:00, 52.7MB/s]


✅ Dataset downloaded and extracted successfully!


In [ ]:
print("Sample files:", os.listdir(extract_path)[:5])

Sample files: ['2832.PNG', '1443.jpg', '650.PNG', '5.PNG', '1601.jpg']


Cloning into 'NationalCard-ImageProccessing'...
remote: Enumerating objects: 489, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 489 (delta 30), reused 7 (delta 5), pack-reused 432 (from 2)
Receiving objects: 100% (489/489), 110.49 MiB | 28.35 MiB/s, done.
Resolving deltas: 100% (119/119), done.


# Set model and path

In [ ]:
detector = FaceAnalysis(name='buffalo_l')
detector.prepare(ctx_id=0, det_size=(640, 640))
objects_model = YOLO('/content/drive/MyDrive/models/TextDetection.pt') # set yolo model for object detection
card_model = YOLO('/content/drive/MyDrive/models/CardDetection.pt') # set yolo model for card detection
hezar = Model.load("hezarai/crnn-fa-printed-96-long") # set hezar ocr model
path = '/content/drive/MyDrive/models/hezar'
hezar.save(path)
hezar_ocr = Model.load(path)
rotate_idcard_model = YOLO('/content/drive/MyDrive/models/RotateIDCard.pt')

/usr/local/lib/python3.11/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


# Rotation Code

In [ ]:
def rotation(path):
    image = cv2.imread(path) # set image
    results = card_model(image) # run card yolo model for image

    if len(results[0].boxes) == 0: # if cant recognize card
        print("No card detected!")
        return None

    # set 4 corner location
    x1, y1, x2, y2 = map(int, results[0].boxes[0].xyxy[0])
    cropped_card = image[y1:y2, x1:x2]

    # detect face in the cropped card
    faces = detector.get(cropped_card)

    # check for faces
    if faces:
        x1, y1, x2, y2 = map(int, faces[0]['bbox'])  # Get face bounding box
        w, h = x2 - x1, y2 - y1
        face_center = (x1 + w // 2, y1 + h // 2)  # Face center
        card_center = (cropped_card.shape[1] // 2, cropped_card.shape[0] // 2)  # Card center

        # calculate rotation angle
        angle = np.arctan2(face_center[1] - card_center[1], face_center[0] - card_center[0]) * (180 / np.pi) + 180
        rotation_matrix = cv2.getRotationMatrix2D(card_center, angle, 1.0)
        rotated = cv2.warpAffine(cropped_card, rotation_matrix, (cropped_card.shape[1], cropped_card.shape[0]))

        return rotated  # return image

    return cropped_card  # if cant find face


In [ ]:
def rotate_idcard(image_path):
  import cv2
  import math
  import numpy as np
  from google.colab.patches import cv2_imshow

  image = cv2.imread(image_path) # set image
  results = card_model(image) # run card yolo model for image

  if len(results[0].boxes) == 0: # if cant recognize card
      print("No card detected!")
      return None

    # set 4 corner location
  x1, y1, x2, y2 = map(int, results[0].boxes[0].xyxy[0])
  cropped_card = image[y1:y2, x1:x2]


  r = rotate_idcard_model.predict(source=cropped_card, show=False, conf=0.5)

  result = r[0]
  boxes = result.boxes.xyxy.cpu().numpy().astype(int)
  confidences = result.boxes.conf.cpu().numpy()
  class_ids = result.boxes.cls.cpu().numpy().astype(int)
  class_names = result.names

  image = cropped_card

  cv2_imshow(image)

  flag_x1, flag_y1, flag_x2, flag_y2 = boxes[0]
  flag_center_x = (flag_x1 + flag_x2) // 2
  flag_center_y = (flag_y1 + flag_y2) // 2


  flag_class_id = class_ids[0]
  flag_class_name = class_names[flag_class_id]


  logo_x1, logo_y1, logo_x2, logo_y2 = boxes[1]
  logo_center_x = (logo_x1 + logo_x2) // 2
  logo_center_y = (logo_y1 + logo_y2) // 2

  logo_class_id = class_ids[1]
  logo_class_name = class_names[logo_class_id]

  cv2.rectangle(image, (flag_x1, flag_y1), (flag_x2, flag_y2), (0, 255, 0), 2)
  cv2.putText(image, flag_class_name, (flag_x1, flag_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
  cv2.circle(image, (flag_center_x, flag_center_y), 5, (0, 255, 0), -1)

  cv2.rectangle(image, (logo_x1, logo_y1), (logo_x2, logo_y2), (255, 0, 0), 2)
  cv2.putText(image, logo_class_name, (logo_x1, logo_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
  cv2.circle(image, (logo_center_x, logo_center_y), 5, (255, 0, 0), -1)


  slope = (logo_center_y - flag_center_y) / (logo_center_x - flag_center_x)
  angle = np.arctan(slope)
  rotation_angle = np.degrees(angle)

  center = (image.shape[1] // 2, image.shape[0] // 2)


  rotation_matrix = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)
  rotated_image = cv2.warpAffine(image, rotation_matrix, (image.shape[1], image.shape[0]))


  cv2_imshow(rotated_image)
  return(rotated_image)

# Use other model for text detection (Hezar)

In [ ]:
def process_img(img):
    data = {}
    results = objects_model(img) # set part of object detection yolo model

    # dict for each box conf
    box_confidences = {}

    for result in results:
        boxes = result.boxes
        for box in boxes:
            xyxy = box.xyxy[0]
            x1, y1, x2, y2 = map(int, xyxy.tolist()) # set corner locations
            label = result.names[int(box.cls)] # set label
            conf = float(box.conf)  # set conf

            cropped_img = img[max(0, y1-3):y2+3, max(0, x1-3):x2+3] # cropped image location (with more nums for better conf)
            cv2_imshow(cropped_img)

            ocr_result = hezar_ocr.predict(cropped_img)  # read text with ocr

            if ocr_result:  # check if OCR returned results
                text_list = []
                for item in ocr_result:
                    text_list.append(item['text'])
                text = " ".join(text_list)  # retrun text
            else:
                text = "" # return empty str

            # best conf text (if have more than one)
            if label in box_confidences:
                if conf > box_confidences[label][1]:
                    box_confidences[label] = (text, conf)
            else:
                box_confidences[label] = (text, conf)

    # print label, text and conf
    for label, (text, conf) in box_confidences.items():
        print(f"Label: {label}, Text: {text}, Confidence: {conf}")
        data[label] = text.strip()

    return data

# Call function and use it

In [ ]:
def detect(folder):
    detected = [] # set empty list
    for img in os.listdir(folder):
      # get images from folder and set csv columns
        if img.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.heic')):
            img_path = os.path.join(folder, img)
            data = {'image_id': img.split('.')[0], 'national_id': '', 'first_name': '', 'last_name': '',
                    'birth_year': '', 'birth_month': '', 'birth_day': '', 'father_name': '',
                    'expiry_year': '', 'expiry_month': '', 'expiry_day': ''}

            # rotation and crop card
            card = rotate_idcard(img_path)
            if card is None:
                continue

            # extract data from cropped image
            extracted_data = process_img(card)


            # set data in dict and in csv
            for key, value in extracted_data.items():
                value = re.sub(r'\s+', ' ', value).strip() # delete space from value (we have no white space in texts)
                if key == 'Expire': # set expiry date value in dict
                    try:
                        y, m, d = value.split('/') # splite to year, month and day
                        data['expiry_year'], data['expiry_month'], data['expiry_day'] = y, m, d # set expiry day, month and year
                    except ValueError:# if cant detect "/" well and get error
                        print(f"Error splitting Expire: {value}")
                elif key == 'Birth': # set birth date value in dict
                    try:
                        y, m, d = value.split('/') # splite to year, month and day
                        data['birth_year'], data['birth_month'], data['birth_day'] = y, m, d # set birth day, month and year
                    except ValueError: # if cant detect "/" well and get error
                        print(f"Error splitting Birth: {value}")
                elif key == 'National': # set national id value in dict
                    data['national_id'] = value
                elif key == 'Name': # set name value in dict
                    data['first_name'] = value
                elif key == 'LastName': # set last name value in dict
                    data['last_name'] = value
                elif key == 'FatherName': # set father name value in dict
                    data['father_name'] = value

            detected.append(data)

    pd.DataFrame(detected).to_csv('image_phase2.csv', index=False, encoding='utf-8')
    return detected

In [ ]:
detect('/content/drive/MyDrive/images')

# Sort csv

In [ ]:
df = pd.read_csv('image_phase2.csv')
df = df.sort_values(by='image_id')
df.to_csv("image_phase2.csv", index=False)

In [ ]:
df

,image_id,national_id,first_name,last_name,birth_year,birth_month,birth_day,father_name,expiry_year,expiry_month,expiry_day
6,1,۶۰۲۴۸۴۰۳۹۶,سراهک,شو!بی,۱۶۰,۰۵,۲,ا,NaN,NaN,NaN
4,20,۸۹۰۰۷۷۱۸۸۰۹,۱۰,۱۱۷۱۰۰۰/۰,۱۱,۰۰۰,۱۷۰۰۱۱,اهم,۱۰۸,۸۰,۰۰۰۱
0,46,۰۷۰۸۱۵۱۰۷۱۰۷,متای,اعهنا,۵۰۱,۷۰,۱۷۱۱,ا/،ا,۱۰,۰۱,۰-۱۱
2,98,۳۶۳۱۶۹۸۳۵۸,ساحل,روانیخش,۱۳۶۵,۰۵,۲۱,بهمن,۱۴۰۹,۱۰,۲۱
1,105,۹۴۹۰۵۴۹۰۰۸,اکبر,کرمی,۱۳۶۷,۱۱,۲۳,تیرداد,۱۴۱۰,۰۳,۲۰
5,1591,۵۹۵۹۰۶۶۳۸۹,مصطفی,بحتاری,۱۳۸۲,۰۶,۰۹,عحعدعلی,۱۴۰۷,۰۷,۱۷
3,2145,۲۴۷۰۹۴۹۷۱۱,ترانه,هارابی,NaN,NaN,NaN,آذرباد,۱۴۰۵,۰۶,۲۶


In [ ]:
df

,image_id,national_id,first_name,last_name,birth_year,birth_month,birth_day,father_name,expiry_year,expiry_month,expiry_day
6,1,۶۰۲۴۸۴۰۳۹۶,سراهک,شو!بی,۱۶۰,۰۵,۲,ا,NaN,NaN,NaN
4,20,۸۹۰۰۷۷۱۸۸۰۹,۱۰,۱۱۷۱۰۰۰/۰,۱۱,۰۰۰,۱۷۰۰۱۱,اهم,۱۰۸,۸۰,۰۰۰۱
0,46,۰۷۰۸۱۵۱۰۷۱۰۷,متای,اعهنا,۵۰۱,۷۰,۱۷۱۱,ا/،ا,۱۰,۰۱,۰-۱۱
2,98,۳۶۳۱۶۹۸۳۵۸,ساحل,روانیخش,۱۳۶۵,۰۵,۲۱,بهمن,۱۴۰۹,۱۰,۲۱
1,105,۹۴۹۰۵۴۹۰۰۸,اکبر,کرمی,۱۳۶۷,۱۱,۲۳,تیرداد,۱۴۱۰,۰۳,۲۰
5,1591,۵۹۵۹۰۶۶۳۸۹,مصطفی,بحتاری,۱۳۸۲,۰۶,۰۹,عحعدعلی,۱۴۰۷,۰۷,۱۷
3,2145,۲۴۷۰۹۴۹۷۱۱,ترانه,هارابی,NaN,NaN,NaN,آذرباد,۱۴۰۵,۰۶,۲۶
